# Tree-Based Models

Unlike the previous notebook, this stage of the project is devided into two separate notebooks based on the preprocessing pipeline required by each model. RandomForest and XGBoost, both rely on the same preprocessing approach and are therefore evaluated together in this notebook. CatBoost, however, uses a different preprocessing strategy, by handling categorical features natively and is evaluated separately in the next notebook.

Raandom Forest serves as the baseline tree-based model for both notebooks. Its evaluated metrics are saved to a CSV file so they can be reused in the CatBoost notebook, allowing all tree-based models to be compared consistently without retraining the baseline model.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 500)

sys.path.append(str(Path().cwd().parent.resolve()))

import preprocessing.features as features

builder = features.FeatureBuilder()

df = pd.read_csv(features.DATASET_PATH)

df_copy = builder.get_df(df)

df_copy.shape

/home/carl/notebooks/airbnb_prices_prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(74111, 27)

In [2]:
import xgboost

xgboost.__version__

'3.3.0'

In [3]:
import sklearn

sklearn.__version__

'1.9.0'

# Baseline Model. RandomForest

In [4]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 25), (14823, 25))

In [6]:
import preprocessing.tree_preprocessor as preprocessor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

cat_features = X_train.select_dtypes(include=['string', 'object']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

tree_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

tree_pipeline.fit(X_train, y_train)

y_pred_test_log = tree_pipeline.predict(X_test)
y_pred_train_log = tree_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 52.22$ | Train MAE: 21.08$
Test RMSE: 118.45$ | Train RMSE: 56.19$
Test R2 Score: 0.68 | Train R2 Score: 0.95


## RandomForest Conclusion

The baseline RandomForest model exhibit significant overfitting, with substantially better performance on the training set that on the test set. Therefore, its current evaluation metrics are not suitable as the primary baseline for comparing tree-based models. In the next step, hyperparameter tuning will be performed using RandomizedSearchCV to reduce overfitting and establish a more reliable baseline for subsequent comparisons. 

# RandomForest. Hyperparameter Tuning.

In [7]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

BASELINE_RANDOM_SEARCH_MODEL = ARTIFACTS_DIR / "random_forest_random_search.joblib"

In [8]:
%%time

import joblib
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

if BASELINE_RANDOM_SEARCH_MODEL.exists():
    print("Loading RandomizedSearchCV...")
    random_search = joblib.load(BASELINE_RANDOM_SEARCH_MODEL)
else:
    print("Training RandomizedSearchCV...")
    
    tree_pipeline = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ])
    
    param_dist = {
        "model__n_estimators": randint(150, 600),
        "model__max_depth": [10, 15, 20, 25, 30],
        "model__min_samples_split": randint(10, 40),
        "model__min_samples_leaf": randint(3, 15),
        "model__max_features": ['log2', 'sqrt', 0.3, 0.3],
        "model__ccp_alpha": [0.0, 0.0001, 0.0005, 0.001]
    }
    
    random_search = RandomizedSearchCV(
        estimator=tree_pipeline,
        param_distributions=param_dist,
        n_iter=15,
        scoring="neg_root_mean_squared_error",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=2
    )

    random_search.fit(X_train, y_train)
    joblib.dump(random_search, BASELINE_RANDOM_SEARCH_MODEL)

best_rf = random_search.best_estimator_
random_search.best_params_

Loading RandomizedSearchCV...
CPU times: user 63.9 ms, sys: 40.1 ms, total: 104 ms
Wall time: 104 ms


{'model__ccp_alpha': 0.0,
 'model__max_depth': 25,
 'model__max_features': 0.3,
 'model__min_samples_leaf': 10,
 'model__min_samples_split': 12,
 'model__n_estimators': 299}

In [9]:
y_pred_test_log = best_rf.predict(X_test)
y_pred_train_log = best_rf.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 53.55$ | Train MAE: 46.96$
Test RMSE: 121.80$ | Train RMSE: 108.13$
Test R2 Score: 0.67 | Train R2 Score: 0.74


In [10]:
results_df = pd.DataFrame(
    columns=['MAE', 'RMSE', 'R2']
)

results_df.loc['RandomForest (baseline, optimized)'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.8,0.67


## RandomForest. Hypyerparameter tuning Conclusion.

The primary objective of hyperparameter tuning was not to maximize predictive performance, but to reduce overfitting and obtain a more stable baseline model for comparison with other algorithms. This objective was sucessfully achieved by shifting the hyperparameter search toward stronger regularization, which significantly reduced the gap between the training and test performance. Although a moderate degree of overfitting still remains, it is acceptable for the purposes of this project and provides a reliable baseline for evaluating more advanced tree-based methods.

To avoid retraining during subsequent executions of the notebook, the optimized model was saved in the `./artifacts/` directory and can be loaded directly when needed. This reduce both computational cost and notebook execution time, while ensuring reproducible results.

# XGBoost

## XGBoost Baseline

In [11]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 25), (14823, 25))

In [13]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

BASELINE_XGBOOST = ARTIFACTS_DIR / "baseline_xgboost.joblib"

In [14]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

In [15]:
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

def objective(trial):

    n_estimators = trial.suggest_int(
        "n_estimators",
        100, 600
    )

    max_depth = trial.suggest_int(
        "max_depth",
        3, 10
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        0.01, 0.3,
        log=True
    )

    subsample = trial.suggest_float(
        "subsample",
        0.5, 1.0
    )

    colsample_bytree = trial.suggest_float(
        "colsample_bytree",
        0.5, 1.0
    )

    gamma = trial.suggest_float(
        "gamma",
        0, 15
    )

    min_child_weight = trial.suggest_int(
        "min_child_weight",
        1, 10
    )

    reg_alpha = trial.suggest_float(
        "reg_alpha",
        1e-4, 10,
        log=True
    )

    reg_lambda = trial.suggest_float(
        "reg_lambda",
        1e-4, 10,
        log=True
    )

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        gamma=gamma,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,

        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )

    pipeline = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train, y_train,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    return -scores.mean()

In [16]:
%%time

if BASELINE_XGBOOST.exists():
    print("Loading XGBoost Baseline Model...")
    xgboost_pipeline = joblib.load(BASELINE_XGBOOST)
else:
    print("Training XGboost Baseline Model...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    study = optuna.create_study(
        direction="minimize"
    )
    
    study.optimize(
        objective,
        n_trials=500
    )
    
    best_params = study.best_params
    xgboost = XGBRegressor(
        **best_params,
        random_state=42
    )

    xgboost_pipeline = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", xgboost)
    ])

    xgboost_pipeline.fit(X_train, y_train)
    joblib.dump(xgboost_pipeline, BASELINE_XGBOOST)

xgboost_pipeline.named_steps['model'].get_params()

Loading XGBoost Baseline Model...
CPU times: user 21.4 ms, sys: 95.2 ms, total: 117 ms
Wall time: 110 ms


{'objective': 'reg:squarederror',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': 0.5646464109315494,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': True,
 'eval_metric': None,
 'feature_types': None,
 'feature_weights': None,
 'gamma': 0.21609777434035493,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': 0.0606349899143156,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': 10,
 'max_leaves': None,
 'min_child_weight': 4,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': 572,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': 42,
 'reg_alpha': 0.0004794840845861772,
 'reg_lambda': 4.0265179785677665,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': 0.6755007870476858,
 'tree_method': None,
 'vali

In [17]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 49.39$ | Train MAE: 38.59$
Test RMSE: 112.34$ | Train RMSE: 87.33$
Test R2 Score: 0.71 | Train R2 Score: 0.82
